## ✅ Import Libraries

In [27]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy torch

In [28]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from google.colab import drive


print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Torch version: 2.10.0+cu128
GPU available: True


In [29]:
!ls /content/Fatocheck/data/processed


ls: cannot access '/content/Fatocheck/data/processed': No such file or directory


In [30]:
!git clone https://github.com/Susanta2025-lab/Fatocheck.git

fatal: destination path 'Fatocheck' already exists and is not an empty directory.


In [31]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## ✅ STEP 2 — Load Cleaned Dataset and Data Preparation

In [32]:
# Load the cleaned news dataset
df = pd.read_csv("/content/drive/MyDrive/Fatocheck/cleaned_news.csv")

df.head()

,content,label,char_length,word_length,clean_content
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,0,5180,889,law enforcement on high alert following threat...
1,Did they post their votes for Hillary already?,0,46,8,did they post their votes for hillary already
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,0,354,52,unbelievable obamas attorney general says most...
3,"Bobby Jindal, raised Hindu, uses story of Chri...",1,8116,1337,bobby jindal raised hindu uses story of christ...
4,SATAN 2: Russia unvelis an image of its terrif...,0,2012,345,satan russia unvelis an image of its terrifyin...


In [33]:
df.isnull().sum()

,0
content,0
label,0
char_length,0
word_length,0
clean_content,48


In [34]:

df=df[["clean_content", "label"]].dropna()

In [35]:
df.isnull().sum()

,0
clean_content,0
label,0


In [36]:
df=df.rename(columns={"clean_content": "text"})

In [37]:
df.head()

,text,label
0,law enforcement on high alert following threat...,0
1,did they post their votes for hillary already,0
2,unbelievable obamas attorney general says most...,0
3,bobby jindal raised hindu uses story of christ...,1
4,satan russia unvelis an image of its terrifyin...,0


## ✅ STEP 3 — Train-Test-Validation Split

In [38]:
# Split the dataset into training, validation, and test sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

## ✅ STEP 4 — Tokenizer + Dataset Conversion

In [39]:
# Prepare the datasets for the transformer model
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [40]:
# Tokenization function
def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding="max_length",
        truncation=True,
        max_length=256
        )

# Apply tokenization to the datasets
train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

#rename the label column to labels
train_ds = train_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")

# Set the format for PyTorch
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/50902 [00:00<?, ? examples/s]

Map:   0%|          | 0/6363 [00:00<?, ? examples/s]

Map:   0%|          | 0/6363 [00:00<?, ? examples/s]

## ✅ STEP 5 — Load BERT

In [41]:
# Load the pre-trained BERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
    )

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## STEP 6 — Metrics

In [42]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }


## ✅ STEP 7 — Training Setup

In [44]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="models/trained/distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

## 🚀 STEP 8 — Train Model

In [45]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.060852,0.045301,0.989470,0.990389
2,0.014915,0.044340,0.991671,0.992369


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=12726, training_loss=0.061978113327164255, metrics={'train_runtime': 5422.4328, 'train_samples_per_second': 18.775, 'train_steps_per_second': 2.347, 'total_flos': 1.339287893993472e+16, 'train_loss': 0.061978113327164255, 'epoch': 2.0})

## 💾 STEP 9 — Save model

In [46]:
# Save the trained model and tokenizer

save_path = "/content/drive/MyDrive/Fatocheck/bert_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Saved at:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved at: /content/drive/MyDrive/Fatocheck/distilbert_model
